In [1]:
import pandas as pd
from datetime import datetime
from tqdm.auto import tqdm

# Import RAG

In [2]:
import sys
sys.path.append('../scripts')
import rag
import vectors

# Load synthetic questions

In [3]:
df_synth = pd.read_csv('../data/data-synth-question.csv', sep='\t', dtype=str)
df_synth

,pmid,ollama_seed,synthetic_question
0,40247149,0,What role do computer algorithms play in devel...
1,40247149,1,Is it more efficient to use machine learning a...
2,40247149,2,What role do researchers believe software tool...
3,40247149,3,What role do expert systems play in improving ...
4,40247149,4,What is one potential limitation of using fMRI...
...,...,...,...
495,40933682,0,What is a major challenge that autistic studen...
496,40933682,1,Does a lack of visibility and understanding fr...
497,40933682,2,What drives college students with autism to de...
498,40933682,3,What is the primary reason that autistic colle...


# Demo RAG

In [4]:
# demonstrate RAG for one question
demo_query = df_synth.iloc[0]['synthetic_question']
print(demo_query)
print()
print(datetime.now())
demo_answer = rag.rag(demo_query, \
do_vector_search=False, num_results=2, model_handle_llm='llama3.2:1b', seed=42)
print(demo_answer)
print(datetime.now())

What role do computer algorithms play in developing effective fMRI features for diagnosing autism spectrum disorders?

2025-09-12 22:53:49.355849
Based on the context provided by the papers from PubMed, it appears that computer algorithms play a significant role in developing effective fMRI (functional magnetic resonance imaging) features for diagnosing autism spectrum disorders (ASD). Here's a summary of the relevant findings:

The first paper, "Functional upper-extremity movements in autism: A narrative literature review" by Shanan Sun et al. (2024), published in Research in Autism Spectrum Disorders (pmid: 40821657), suggests that fMRI can be used to identify differences in upper extremity motor skills among autistic individuals. The authors found that fMRI features, such as connectivity and functional networks, may be altered in autistic individuals compared to healthy controls.

The second paper, "Exploring the effects of age and sex on sensory sensitivities in middle and older ag

# Generate answers by RAG

## Function

In [5]:
# common parameters for RAG
print(rag.config['do_vector_search'])
print(rag.config['num_results'])

False
5


In [6]:
def generate_answers(synth_records, model_handle_llm, seed):
    answers = []
    for record in tqdm(synth_records):
        query = record['synthetic_question']
        answer = rag.rag(query, \
        do_vector_search=rag.config['do_vector_search'], num_results=rag.config['num_results'], \
        model_handle_llm=model_handle_llm, seed=seed)
        pmid = record['pmid']
        ollama_seed_question = record['ollama_seed']
        answer_dict = {'pmid' : pmid, 'ollama_seed' : ollama_seed_question, \
        'answer_'+model_handle_llm : answer}
        answers.append(answer_dict)
    return pd.DataFrame.from_records(answers)

## Work

In [7]:
# sample size, dictionary format for dataset
sample_size = 30
synth_records = df_synth.to_dict('records')[:sample_size]
print(len(synth_records))

30


In [8]:
# generate answers for llama3.2
print(datetime.now())
answers_llama = generate_answers(synth_records=synth_records, \
model_handle_llm='llama3.2:1b', seed=42)
print(datetime.now())
answers_llama

2025-09-12 22:54:18.297249


  0%|          | 0/30 [00:00<?, ?it/s]

2025-09-12 23:21:24.207063


,pmid,ollama_seed,answer_llama3.2:1b
0,40247149,0,Based on the provided context and papers from ...
1,40247149,1,Based on the context provided by the papers fr...
2,40247149,2,Researchers believe that software tools play a...
3,40247149,3,Based on the context provided by the papers yo...
4,40247149,4,One potential limitation of using functional m...
5,40266512,0,Based on the provided context and PubMed paper...
6,40266512,1,Based on the context provided by the papers fr...
7,40266512,2,Based on the provided context and papers from ...
8,40266512,3,Based on the provided context and papers from ...
9,40266512,4,Based on the context provided by the papers fr...


In [9]:
# how long are the answers for llama3.2?
print(answers_llama['answer_llama3.2:1b'].apply(lambda x : \
len(x.split())).quantile([0.5, 0.75, 0.99, 1.0])) # percentiles

0.50    289.50
0.75    335.25
0.99    418.20
1.00    424.00
Name: answer_llama3.2:1b, dtype: float64


In [10]:
# generate answers for gemma3
print(datetime.now())
answers_gemma = generate_answers(synth_records=synth_records, \
model_handle_llm='gemma3:1b', seed=42)
print(datetime.now())
answers_gemma

2025-09-12 23:21:24.245866


  0%|          | 0/30 [00:00<?, ?it/s]

2025-09-12 23:49:01.762012


,pmid,ollama_seed,answer_gemma3:1b
0,40247149,0,"Okay, here's an answer based on the provided t..."
1,40247149,1,"Okay, let’s tackle this question based on the ..."
2,40247149,2,"Okay, based on the provided context and papers..."
3,40247149,3,"Okay, let’s analyze this context and answer yo..."
4,40247149,4,"Okay, here’s an analysis of the provided text,..."
5,40266512,0,"Okay, based on the provided context and the pr..."
6,40266512,1,"Okay, here's an analysis of the provided text,..."
7,40266512,2,"Okay, let’s analyze the provided papers concer..."
8,40266512,3,"Okay, here's an answer based on the provided c..."
9,40266512,4,"Okay, let’s analyze the provided papers focusi..."


In [11]:
# how long are the answers for gemma3?
print(answers_gemma['answer_gemma3:1b'].apply(lambda x : \
len(x.split())).quantile([0.5, 0.75, 0.99, 1.0])) # percentiles

0.50    402.50
0.75    471.75
0.99    637.81
1.00    641.00
Name: answer_gemma3:1b, dtype: float64


In [12]:
# put together answers from different models
df_synth_answer = answers_llama.merge(answers_gemma, on=['pmid', 'ollama_seed'], how='inner')
df_synth_answer

,pmid,ollama_seed,answer_llama3.2:1b,answer_gemma3:1b
0,40247149,0,Based on the provided context and papers from ...,"Okay, here's an answer based on the provided t..."
1,40247149,1,Based on the context provided by the papers fr...,"Okay, let’s tackle this question based on the ..."
2,40247149,2,Researchers believe that software tools play a...,"Okay, based on the provided context and papers..."
3,40247149,3,Based on the context provided by the papers yo...,"Okay, let’s analyze this context and answer yo..."
4,40247149,4,One potential limitation of using functional m...,"Okay, here’s an analysis of the provided text,..."
5,40266512,0,Based on the provided context and PubMed paper...,"Okay, based on the provided context and the pr..."
6,40266512,1,Based on the context provided by the papers fr...,"Okay, here's an analysis of the provided text,..."
7,40266512,2,Based on the provided context and papers from ...,"Okay, let’s analyze the provided papers concer..."
8,40266512,3,Based on the provided context and papers from ...,"Okay, here's an answer based on the provided c..."
9,40266512,4,Based on the context provided by the papers fr...,"Okay, let’s analyze the provided papers focusi..."


# Write CSV file

In [13]:
# write CSV file
df_synth_answer.to_csv('../data/data-synth-answer.csv', index=False, sep='\t')

In [14]:
print(datetime.now())

2025-09-12 23:49:01.829765
